# z603 - LightGBM Baseline (Etapa 3)
Features: solo Etapa 2 (`tb_features_FE601.parquet`). Target: `log1p(tn)`, dos periodos adelante (la competencia predice 202002 con datos hasta 201912).

In [1]:
import os
import numpy as np
import polars as pl
import lightgbm as lgb
import warnings
warnings.filterwarnings("ignore")

In [2]:
PARAM = {
    'experimento': 'LGB01',
    'kaggle_competition': 'labo-iii-2026-ba',
    'base_path': './exp/FE601/',
    'archivo_features': 'tb_features_FE601.parquet',
    'apredecir_path': './datasets/product_id_apredecir201912.txt',
    'horizonte_meses': 2,          # la competencia predice 2 meses adelante del ultimo dato disponible
    'periodo_ultimo_dato': 201912,
    'periodo_target_final': 202002,
    'semilla': 102103
}

ruta = os.path.join('./exp', PARAM['experimento'])
os.makedirs(ruta, exist_ok=True)
print(ruta)

./exp/LGB01


## 1. Cargar features y armar el target a horizonte 2
El modelo se entrena para predecir `tn` de DOS periodos adelante, parado en cada periodo `p` (usando solo lo que se sabe hasta `p`). Asi, en 201912 predice directamente 202002 -- mismo horizonte con el que se entrena.

In [3]:
def periodo_a_meses(periodo: int) -> int:
    return (periodo // 100) * 12 + (periodo % 100)

df = pl.read_parquet(os.path.join(PARAM['base_path'], PARAM['archivo_features']))
df = df.sort(["product_id", "periodo"])

H = PARAM['horizonte_meses']

# target = tn del propio producto, H periodos adelante (shift negativo en el tiempo)
df = df.with_columns(
    pl.col("tn").shift(-H).over("product_id").alias("tn_target")
)
df = df.with_columns(
    (pl.col("periodo_m") + H).alias("periodo_target_m")
)

## 2. Split train / valid
Definido por el periodo TARGET (lo que se predice), no por el periodo de las features:
- train: target &le; 201910
- valid: target entre 201911 y 201912

Filas sin target (fuera del rango de vida del producto) se descartan.

In [4]:
m_201910 = periodo_a_meses(201910)
m_201911 = periodo_a_meses(201911)
m_201912 = periodo_a_meses(201912)

df_valido = df.filter(pl.col("tn_target").is_not_null())

train = df_valido.filter(pl.col("periodo_target_m") <= m_201910)
valid = df_valido.filter(
    (pl.col("periodo_target_m") >= m_201911) & (pl.col("periodo_target_m") <= m_201912)
)

print("train:", train.height, " valid:", valid.height)

train: 27249  valid: 1827


## 3. Preparar matrices
Se excluyen columnas identificadoras/target/intermedias. `product_id` queda como feature categorica (LightGBM la maneja nativamente).

In [5]:
cols_excluir = {"tn", "tn_target", "tn_shift1", "periodo", "periodo_target_m", "nacimiento_m"}
features = [c for c in df.columns if c not in cols_excluir]
categoricas = ["product_id"]

def a_pandas(tabla):
    pdf = tabla.select(features + ["tn_target"]).to_pandas()
    for c in categoricas:
        pdf[c] = pdf[c].astype("category")
    return pdf

train_pd = a_pandas(train)
valid_pd = a_pandas(valid)

X_train = train_pd[features]
y_train = np.log1p(train_pd["tn_target"].clip(lower=0))

X_valid = valid_pd[features]
y_valid = np.log1p(valid_pd["tn_target"].clip(lower=0))

## 4. Entrenar LightGBM (hiperparametros fijos, sin tuning)

In [6]:
params = {
    'objective': 'regression',
    'metric': 'rmse',
    'learning_rate': 0.05,
    'num_leaves': 31,
    'min_data_in_leaf': 50,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 1,
    'verbosity': -1,
    'seed': PARAM['semilla']
}

dtrain = lgb.Dataset(X_train, label=y_train, categorical_feature=categoricas)
dvalid = lgb.Dataset(X_valid, label=y_valid, categorical_feature=categoricas, reference=dtrain)

modelo = lgb.train(
    params,
    dtrain,
    num_boost_round=2000,
    valid_sets=[dtrain, dvalid],
    valid_names=['train', 'valid'],
    callbacks=[lgb.early_stopping(stopping_rounds=100), lgb.log_evaluation(period=100)]
)

print("mejor iteracion:", modelo.best_iteration)

Training until validation scores don't improve for 100 rounds
[100]	train's rmse: 0.44506	valid's rmse: 0.558545
[200]	train's rmse: 0.389724	valid's rmse: 0.557722
[300]	train's rmse: 0.361465	valid's rmse: 0.556625
Early stopping, best iteration is:
[291]	train's rmse: 0.363402	valid's rmse: 0.555413
mejor iteracion: 291


## 5. Prediccion para 202002
Se usan las features paradas en 201912 (el ultimo dato real). El modelo, entrenado a horizonte 2, predice directamente 202002.

In [7]:
futuro = df.filter(pl.col("periodo") == PARAM['periodo_ultimo_dato'])
futuro_pd = futuro.select(features).to_pandas()
for c in categoricas:
    futuro_pd[c] = futuro_pd[c].astype("category")

pred_log = modelo.predict(futuro_pd, num_iteration=modelo.best_iteration)
pred_tn = np.expm1(pred_log)
pred_tn = np.clip(pred_tn, 0, None)

resultado = futuro.select(["product_id"]).to_pandas()
resultado["tn"] = pred_tn

## 6. Filtrar a productos a predecir y armar submit
Verificar el formato exacto contra `sample_submission.csv` de Kaggle antes de subir, si esta disponible -- columnas asumidas: `product_id, tn`.

In [8]:
apredecir = pl.read_csv(PARAM['apredecir_path'], separator="\t").to_pandas()

submit = apredecir[["product_id"]].merge(resultado, on="product_id", how="left")
print("nulos en submit (deberian ser 0):", submit["tn"].isna().sum())
submit["tn"] = submit["tn"].fillna(0.0)

archivo_submit = os.path.join(ruta, f"{PARAM['experimento']}_submit.csv")
submit.to_csv(archivo_submit, index=False)
print(archivo_submit)
submit.head()

nulos en submit (deberian ser 0): 0
./exp/LGB01/LGB01_submit.csv


,product_id,tn
0,20001,1512.709168
1,20002,1242.054716
2,20003,849.503088
3,20004,661.943440
4,20005,593.411877


## 7. Submit a Kaggle

In [9]:
def kaggle_submit(competencia, archivo, mensaje):
    comando = f'kaggle competitions submit -c {competencia} -f {archivo} -m "{mensaje}"'
    os.system(comando)

kaggle_submit(PARAM['kaggle_competition'], archivo_submit, f"{PARAM['experimento']} baseline LightGBM Etapa 2")

100%|██████████| 18.7k/18.7k [00:00<00:00, 58.8kB/s]


99 submissions remaining today.
Successfully submitted to Labo III, 2026 BA